A/B Hypothesis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Beautiful plot style
sns.set(style="whitegrid", palette="viridis", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported!")

# Show current folder
print("Current working directory:", os.getcwd())

# === DVC Pull ===
print("\nPulling data with DVC...")
!dvc pull --quiet
print("✅ DVC pull completed")

#Load data
data_path = "../data/insurance_data.csv"

print(f"Loading data from: {data_path} ... (this may take 15-25 seconds)")

df = pd.read_csv(
    data_path,
    sep='|',
    low_memory=True
)

# === Fix European number format (comma as decimal) ===
numeric_cols_to_fix = ['CapitalOutstanding', 'CustomValueEstimate', 'SumInsured',
                       'CalculatedPremiumPerTerm', 'TotalPremium', 'TotalClaims']

for col in numeric_cols_to_fix:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '.').str.replace(' ', '')
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert date
df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'] + '-01', errors='coerce')

print(f"✅ Data loaded and cleaned successfully! Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("First 3 rows:")
display(df.head(3))

print("\nNumeric columns fixed (comma → dot):", numeric_cols_to_fix)

✅ Libraries imported!
Current working directory: c:\Users\bezaw\OneDrive\Desktop\Git Projects\Insurance-risk-analytics\notebooks

Pulling data with DVC...
✅ DVC pull completed
Loading data from: ../data/insurance_data.csv ... (this may take 15-25 seconds)


C:\Users\bezaw\AppData\Local\Temp\ipykernel_9348\2632712079.py:27: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


✅ Data loaded and cleaned successfully! Shape: 1,000,098 rows × 52 columns
First 3 rows:


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00-01:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00-01:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00-01:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0



Numeric columns fixed (comma → dot): ['CapitalOutstanding', 'CustomValueEstimate', 'SumInsured', 'CalculatedPremiumPerTerm', 'TotalPremium', 'TotalClaims']


Adding new columns inorder to do the hypothesis testing

In [2]:
df['HasClaim']= (df['TotalClaims']>0).astype(int)

df['Margin']= df['TotalPremium'] - df['TotalClaims']

print("New columns added:")
print(df[['HasClaim', 'Margin', 'TotalClaims','TotalPremium']].head())

print("\nHasClaim distribution:\n", df['HasClaim'].value_counts(normalize=True))

New columns added:
   HasClaim      Margin  TotalClaims  TotalPremium
0         0   21.929825          0.0     21.929825
1         0   21.929825          0.0     21.929825
2         0    0.000000          0.0      0.000000
3         0  512.848070          0.0    512.848070
4         0    0.000000          0.0      0.000000

HasClaim distribution:
 HasClaim
0    0.997212
1    0.002788
Name: proportion, dtype: float64


1. H0= There is no risk difference across province

In [3]:
print("=== HYPOTHESIS 1: There are no risk differences across provinces ===")

# Select two provinces (as per challenge instructions)
province_a = 'Western Cape'   # low-risk from EDA
province_b = 'Gauteng'        # highest-risk from EDA

# Create two groups
group_a = df[df['Province'] == province_a].copy()
group_b = df[df['Province'] == province_b].copy()

print(f"Group A (Control): {province_a} → {len(group_a):,} policies")
print(f"Group B (Test): {province_b} → {len(group_b):,} policies")

# 1. Claim Frequency (proportion of policies with at least one claim)
freq_a = group_a['HasClaim'].mean()
freq_b = group_b['HasClaim'].mean()

print(f"\nClaim Frequency - {province_a}: {freq_a:.4f} ({freq_a*100:.2f}%)")
print(f"Claim Frequency - {province_b}: {freq_b:.4f} ({freq_b*100:.2f}%)")

# Chi-squared test for Claim Frequency
from scipy.stats import chi2_contingency
contingency = pd.crosstab(df['Province'].isin([province_a, province_b]), df['HasClaim'])
chi2, p_freq, dof, expected = chi2_contingency(contingency)
print(f"\nChi-squared p-value for Claim Frequency: {p_freq:.5f}")

# 2. Claim Severity (average claim amount ONLY when claim > 0)
severity_a = group_a[group_a['HasClaim'] == 1]['TotalClaims'].mean()
severity_b = group_b[group_b['HasClaim'] == 1]['TotalClaims'].mean()

print(f"\nClaim Severity - {province_a}: R{severity_a:,.2f}")
print(f"Claim Severity - {province_b}: R{severity_b:,.2f}")

# t-test for Claim Severity
t_stat, p_sev = stats.ttest_ind(
    group_a[group_a['HasClaim']==1]['TotalClaims'],
    group_b[group_b['HasClaim']==1]['TotalClaims'],
    equal_var=False
)
print(f"t-test p-value for Claim Severity: {p_sev:.5f}")

# Decision
alpha = 0.05
print("\n=== CONCLUSION ===")
if p_freq < alpha:
    print(f"✅ REJECT H0: There IS a significant risk difference in claim frequency (p = {p_freq:.5f})")
else:
    print(f"❌ FAIL TO REJECT H0: No significant difference in claim frequency")

if p_sev < alpha:
    print(f"✅ REJECT H0: There IS a significant difference in claim severity (p = {p_sev:.5f})")
else:
    print(f"❌ FAIL TO REJECT H0: No significant difference in claim severity")

=== HYPOTHESIS 1: There are no risk differences across provinces ===


Group A (Control): Western Cape → 170,796 policies
Group B (Test): Gauteng → 393,865 policies

Claim Frequency - Western Cape: 0.0022 (0.22%)
Claim Frequency - Gauteng: 0.0034 (0.34%)

Chi-squared p-value for Claim Frequency: 0.00001

Claim Severity - Western Cape: R28,095.85
Claim Severity - Gauteng: R22,243.88
t-test p-value for Claim Severity: 0.03060

=== CONCLUSION ===
✅ REJECT H0: There IS a significant risk difference in claim frequency (p = 0.00001)
✅ REJECT H0: There IS a significant difference in claim severity (p = 0.03060)


### Hypothesis 1: There are no risk differences across provinces

**Groups compared**:
- Group A (Control): Western Cape (170,796 policies)
- Group B (Test): Gauteng (393,865 policies)

**Results**:
- Claim Frequency: Western Cape = 0.22%, Gauteng = 0.34% → p-value = 0.00001
- Claim Severity: Western Cape = R28,095.85, Gauteng = R22,243.88 → p-value = 0.03060

**Conclusion**:  
**Reject H₀** for both metrics (p < 0.05). There is a statistically significant risk difference across provinces.

**Business Recommendation**:
Gauteng shows higher claim frequency, making it a high-risk region. Western Cape has lower frequency but higher severity when claims occur.  
**Action**: Increase premiums or tighten underwriting in Gauteng. Offer lower premiums in Western Cape to attract more low-risk clients. This helps optimise marketing strategy and discover low-risk segments.

Hypothesis 2 and 3 - ZIP Codes

In [4]:
# ====================== CELL 4: HYPOTHESIS 2 & 3 - ZIP CODES ======================

print("=== HYPOTHESIS 2 & 3: Risk and Margin differences between zip codes ===")

# Step 1: Find two zip codes with enough data (high-risk vs low-risk)
zip_stats = df.groupby('PostalCode').agg({
    'HasClaim': 'mean',           # frequency
    'TotalClaims': 'mean',        # severity
    'Margin': 'mean',             # profit
    'PostalCode': 'size'          # number of policies
}).rename(columns={'PostalCode': 'Count'})

# Filter zip codes that have at least 1000 policies (so test is reliable)
valid_zips = zip_stats[zip_stats['Count'] >= 1000]

# Pick the zip code with highest LossRatio (riskiest) and lowest (safest)
riskiest_zip = valid_zips['HasClaim'].idxmax()
safest_zip   = valid_zips['HasClaim'].idxmin()

print(f"Selected Group A (low-risk zip): {safest_zip}  → {valid_zips.loc[safest_zip, 'Count']:,} policies")
print(f"Selected Group B (high-risk zip): {riskiest_zip} → {valid_zips.loc[riskiest_zip, 'Count']:,} policies")

# Create the two groups
group_a_zip = df[df['PostalCode'] == safest_zip].copy()
group_b_zip = df[df['PostalCode'] == riskiest_zip].copy()

# === Hypothesis 2: Risk differences (same as before) ===
freq_a = group_a_zip['HasClaim'].mean()
freq_b = group_b_zip['HasClaim'].mean()

print(f"\nClaim Frequency - Low-risk zip: {freq_a:.4f} ({freq_a*100:.2f}%)")
print(f"Claim Frequency - High-risk zip: {freq_b:.4f} ({freq_b*100:.2f}%)")

from scipy.stats import chi2_contingency
contingency_zip = pd.crosstab(df['PostalCode'].isin([safest_zip, riskiest_zip]), df['HasClaim'])
_, p_freq_zip, _, _ = chi2_contingency(contingency_zip)
print(f"Chi-squared p-value for Claim Frequency: {p_freq_zip:.5f}")

# Claim Severity
sev_a = group_a_zip[group_a_zip['HasClaim'] == 1]['TotalClaims'].mean()
sev_b = group_b_zip[group_b_zip['HasClaim'] == 1]['TotalClaims'].mean()
print(f"Claim Severity - Low-risk zip: R{sev_a:,.2f}")
print(f"Claim Severity - High-risk zip: R{sev_b:,.2f}")

t_stat_zip, p_sev_zip = stats.ttest_ind(
    group_a_zip[group_a_zip['HasClaim']==1]['TotalClaims'],
    group_b_zip[group_b_zip['HasClaim']==1]['TotalClaims'],
    equal_var=False
)
print(f"t-test p-value for Claim Severity: {p_sev_zip:.5f}")

# === Hypothesis 3: Margin (profit) difference ===
margin_a = group_a_zip['Margin'].mean()
margin_b = group_b_zip['Margin'].mean()
print(f"\nAverage Margin - Low-risk zip: R{margin_a:,.2f}")
print(f"Average Margin - High-risk zip: R{margin_b:,.2f}")

t_stat_margin, p_margin = stats.ttest_ind(group_a_zip['Margin'], group_b_zip['Margin'], equal_var=False)
print(f"t-test p-value for Margin difference: {p_margin:.5f}")

# === Final Decisions ===
alpha = 0.05
print("\n=== CONCLUSIONS ===")
print("H2 - Risk difference:", "REJECT H0" if p_freq_zip < alpha else "FAIL TO REJECT H0")
print("H3 - Margin difference:", "REJECT H0" if p_margin < alpha else "FAIL TO REJECT H0")

=== HYPOTHESIS 2 & 3: Risk and Margin differences between zip codes ===
Selected Group A (low-risk zip): 1830  → 1,129 policies
Selected Group B (high-risk zip): 1030 → 1,163 policies

Claim Frequency - Low-risk zip: 0.0000 (0.00%)
Claim Frequency - High-risk zip: 0.0086 (0.86%)
Chi-squared p-value for Claim Frequency: 0.21732
Claim Severity - Low-risk zip: Rnan
Claim Severity - High-risk zip: R10,814.24
t-test p-value for Claim Severity: nan

Average Margin - Low-risk zip: R46.88
Average Margin - High-risk zip: R-33.47
t-test p-value for Margin difference: 0.08852

=== CONCLUSIONS ===
H2 - Risk difference: FAIL TO REJECT H0
H3 - Margin difference: FAIL TO REJECT H0


C:\Users\bezaw\AppData\Local\Temp\ipykernel_9348\991325454.py:45: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  t_stat_zip, p_sev_zip = stats.ttest_ind(


### Hypothesis 2 & 3: Risk and Margin differences between zip codes

**Groups compared**:
- Low-risk zip (1830): 1,129 policies
- High-risk zip (1030): 1,163 policies

**Results**:
- Claim Frequency: 0.00% vs 0.86% → p-value = 0.21732
- Claim Severity: NaN vs R10,814.24 → p-value = NaN (small sample)
- Margin: R46.88 vs R-33.47 → p-value = 0.08852

**Conclusion**:  
Fail to reject H₀ for both hypotheses (p > 0.05). No statistically significant risk or margin difference between these two zip codes.

**Business Recommendation**:
Zip-code level differences are not strong enough to justify separate pricing. Focus pricing adjustments at the province level (as found in Hypothesis 1). Collect more data or group zip codes by province for future analysis.

Hypothesis 4 - Gender

In [5]:
print("=== HYPOTHESIS 4: There is no significant risk difference between Women and men ===")

# Filter only policies with known gender
gender_data = df[df['Gender'].isin(['Male', 'Female'])].copy()

group_male = gender_data[gender_data['Gender'] == 'Male'].copy()
group_female = gender_data[gender_data['Gender'] == 'Female'].copy()

print(f"Male policies: {len(group_male):,}")
print(f"Female policies: {len(group_female):,}")

# Claim Frequency
freq_male = group_male['HasClaim'].mean()
freq_female = group_female['HasClaim'].mean()

print(f"\nClaim Frequency - Male: {freq_male:.4f} ({freq_male*100:.2f}%)")
print(f"Claim Frequency - Female: {freq_female:.4f} ({freq_female*100:.2f}%)")

# Chi-squared test
contingency_gender = pd.crosstab(gender_data['Gender'], gender_data['HasClaim'])
_, p_freq_gender, _, _ = chi2_contingency(contingency_gender)
print(f"Chi-squared p-value for Claim Frequency: {p_freq_gender:.5f}")

# Claim Severity
sev_male = group_male[group_male['HasClaim'] == 1]['TotalClaims'].mean()
sev_female = group_female[group_female['HasClaim'] == 1]['TotalClaims'].mean()

print(f"Claim Severity - Male: R{sev_male:,.2f}")
print(f"Claim Severity - Female: R{sev_female:,.2f}")

t_stat_gender, p_sev_gender = stats.ttest_ind(
    group_male[group_male['HasClaim']==1]['TotalClaims'],
    group_female[group_female['HasClaim']==1]['TotalClaims'],
    equal_var=False
)
print(f"t-test p-value for Claim Severity: {p_sev_gender:.5f}")

alpha = 0.05
print("\n=== CONCLUSION ===")
print("H4 - Risk difference (frequency):", "REJECT H0" if p_freq_gender < alpha else "FAIL TO REJECT H0")
print("H4 - Risk difference (severity):", "REJECT H0" if p_sev_gender < alpha else "FAIL TO REJECT H0")

=== HYPOTHESIS 4: There is no significant risk difference between Women and men ===
Male policies: 42,817
Female policies: 6,755

Claim Frequency - Male: 0.0022 (0.22%)
Claim Frequency - Female: 0.0021 (0.21%)
Chi-squared p-value for Claim Frequency: 0.95146
Claim Severity - Male: R14,858.55
Claim Severity - Female: R17,874.72
t-test p-value for Claim Severity: 0.56803

=== CONCLUSION ===
H4 - Risk difference (frequency): FAIL TO REJECT H0
H4 - Risk difference (severity): FAIL TO REJECT H0


## Task 3 Summary: A/B Hypothesis Testing

**H1 - Provinces**: Rejected H₀ for both claim frequency (p=0.00001) and severity (p=0.03060).  
**Recommendation**: Increase premiums in Gauteng, lower in Western Cape.

**H2 & H3 - Zip codes**: Failed to reject H₀ for both risk and margin (p>0.05).  
**Recommendation**: Zip-code level pricing not justified; use province level instead.

**H4 - Gender**: Failed to reject H₀ for both metrics (p>0.05).  
**Recommendation**: Gender not a reliable risk driver with current data (too many "Not specified").

**Overall Insight**: Province and vehicle type are the strongest risk drivers. Focus marketing and pricing adjustments on these factors to optimize premiums and attract low-risk clients.
